# Scam Checker: phone test page

Runs the trained model on Kaggle's GPU and opens a web page you can use on your phone.

**Before you run**
1. Right panel → **Add Input** → **Your Work** → add the output of **scam-checker training**
   (it contains `scam_checker_lora`).
2. **Accelerator: GPU T4 x2**, **Internet: On**.
3. **Run All**. After 3–4 minutes the last cell prints a `https://….gradio.live` link: open it on your phone.

The page works only while this session is running, and it uses GPU hours. Stop the session
(power button) when you're done testing.

In [ ]:
%%capture
!pip install -q unsloth gradio

In [ ]:
import glob, os, sys
!rm -rf /kaggle/working/scam-checker && git clone -q --depth 1 https://github.com/Amanseek/scam-checker /kaggle/working/scam-checker
sys.path.insert(0, "/kaggle/working/scam-checker")

# Find the trained adapter in the attached input (folder, or the zip).
found = glob.glob("/kaggle/input/**/scam_checker_lora/adapter_config.json", recursive=True)
if not found:
    for z in glob.glob("/kaggle/input/**/scam_checker_lora.zip", recursive=True):
        !cd /kaggle/working && unzip -oq "{z}"
    found = glob.glob("/kaggle/working/scam_checker_lora/adapter_config.json")
assert found, "Add the 'scam-checker training' output as an input first (right panel → Add Input)."
ADAPTER = os.path.dirname(found[0])
print("adapter:", ADAPTER)

In [ ]:
from serve.checker import Checker, load_generate
checker = Checker(load_generate(ADAPTER))
for m in ["Dear customer your SBI account will be blocked today, update KYC now http://sbi-kyc-update.xyz",
          "Mummy main ghar 8 baje tak aa jaunga, khana rakh dena"]:
    r = checker.check(m)
    print(r["seconds"], "s |", r["raw"].strip())
    assert r["ok"], "Model answer wasn't readable; don't open the page yet."


In [ ]:
from serve.app import build_app
app = build_app(checker, feedback_path="/kaggle/working/feedback.jsonl")
app.queue().launch(share=True, debug=False)
# Open the https://....gradio.live link printed above on your phone.